In [1]:
import os, requests
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import torch
import matplotlib.pyplot as plt

### Download the data

In [2]:
def search_images_serpapi(term, engine="google_images", key=None, max_images=150):
    key = key or os.environ.get("SERPAPI_KEY")
    if not key:
        raise ValueError("Set SERPAPI_KEY or pass key=")
    url, params = "https://serpapi.com/search", dict(engine=engine, q=term, api_key=key)
    res = []
    while url and len(res) < max_images:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        j = r.json()
        res += [o["original"] for o in j.get("images_results", []) if "original" in o]
        url, params = j.get("serpapi_pagination", {}).get("next"), dict(api_key=key)
    return list(dict.fromkeys(res))[:max_images]


def _download_one(args):
    i, url, dest = args
    f = Path(dest) / f"{i:04d}.jpg"
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        f.write_bytes(r.content)
        Image.open(f).convert("RGB").save(f)      # fails if it isn't a real image
        return True
    except Exception:
        f.unlink(missing_ok=True)                 # remove broken files
        return False


def download_images(dest, urls, n_workers=16):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        ok = sum(pool.map(_download_one, [(i, u, dest) for i, u in enumerate(urls)]))
    print(f"{dest}: {ok}/{len(urls)} images saved")

In [3]:
animals = "cat", "dog", "fish", "kangaroo", "pig"
path = Path('animals')
key = os.environ.get('SERPAPI_KEY', 'd834fe07e8b79ba923b1ca87a72d68da70d55d30446f3bc232f204037edb3d3f')

In [4]:
if not path.exists():
    path.mkdir()
    for animal in animals:
        dest = path/animal
        dest.mkdir(exist_ok=True)
        results = search_images_serpapi(animal, engine="google_images", key=key, max_images=250)
        download_images(dest, urls=results)

/opt/homebrew/Caskroom/miniforge/base/envs/tongxinalex/lib/python3.12/site-packages/PIL/Image.py:1134: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


animals/cat: 234/250 images saved
animals/dog: 221/250 images saved
animals/fish: 229/250 images saved
animals/kangaroo: 228/250 images saved
animals/pig: 223/250 images saved


### Load the data

In [5]:
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

In [6]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats, for pretrained models

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),        # min_scale=0.5
    transforms.RandomHorizontalFlip(),
    # transforms.RandomAffine(degrees=10),                        # rotation
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
val_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [8]:
full_train = datasets.ImageFolder("animals", transform=train_tfm)
full_val  = datasets.ImageFolder("animals", transform=val_tfm)

n = len(full_train)
perm = torch.randperm(n, generator=torch.Generator().manual_seed(0))
cut = int(0.8 * n)
train_ds = Subset(full_train, perm[:cut].tolist())
val_ds   = Subset(full_val,  perm[cut:].tolist())

In [10]:
train_dl, val_dl = DataLoader(train_ds, batch_size=256, shuffle=True), DataLoader(val_ds, batch_size=256)

### Train the model

In [11]:
def train_epoch(model, train_dl, loss_func, optimizer):
    model.train()
    losses = []
    for x,y in train_dl:
        # x = x.view(x.size(0), -1)
        # y = y.view(-1)
        preds = model(x)
        loss = loss_func(preds, y)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return sum(losses)/len(losses)

In [12]:
def batch_accuracy(preds, trgts):
    preds = preds.argmax(dim=1)
    correct = preds == trgts
    return correct.float().mean()

In [13]:
def validate_epoch(model, val_dl):
    model.eval()
    with torch.no_grad():
        accs = []
        for x,y in val_dl:
            # x = x.view(x.size(0), -1)
            # y = y.view(-1)
            preds = model(x)
            acc = batch_accuracy(preds, y)
            accs.append(acc)
        return torch.stack(accs).mean()

In [14]:
def train_model(model, train_dl, val_dl, loss_func, optimizer, epochs):
    history = {"epoch": [], "loss": [], "acc": []}
    for epoch in range(epochs):
        loss = train_epoch(model, train_dl, loss_func, optimizer)
        acc = validate_epoch(model, val_dl)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Validation Accuracy: {acc:.4f}")
        history["loss"].append(loss)
        history["acc"].append(acc)
    return history

In [15]:
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
loss_func = nn.CrossEntropyLoss()

In [20]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(full_train.classes))

In [21]:
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

hist_ce1 = train_model(model=model, train_dl=train_dl, val_dl=val_dl, loss_func=loss_func, optimizer=optimizer, epochs=3)

Epoch 1/3, Loss: 1.7430, Validation Accuracy: 0.2907
Epoch 2/3, Loss: 1.3667, Validation Accuracy: 0.6079
Epoch 3/3, Loss: 1.0422, Validation Accuracy: 0.7445


In [22]:
for p in model.parameters():
    p.requires_grad = True
optimizer = torch.optim.Adam([
    {"params": [p for n, p in model.named_parameters() if not n.startswith("fc")], "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-3},
])

hist_ce2 = train_model(model=model, train_dl=train_dl, val_dl=val_dl, loss_func=loss_func, optimizer=optimizer, epochs=10)

Epoch 1/10, Loss: 0.7804, Validation Accuracy: 0.8678
Epoch 2/10, Loss: 0.5214, Validation Accuracy: 0.8899
Epoch 3/10, Loss: 0.3723, Validation Accuracy: 0.9031
Epoch 4/10, Loss: 0.2906, Validation Accuracy: 0.9119
Epoch 5/10, Loss: 0.2278, Validation Accuracy: 0.9163
Epoch 6/10, Loss: 0.1854, Validation Accuracy: 0.9163
Epoch 7/10, Loss: 0.1543, Validation Accuracy: 0.9427
Epoch 8/10, Loss: 0.1275, Validation Accuracy: 0.9339
Epoch 9/10, Loss: 0.1242, Validation Accuracy: 0.9339
Epoch 10/10, Loss: 0.1014, Validation Accuracy: 0.9427


In [32]:
torch.save({
    "state_dict": model.state_dict(),
    "classes": full_train.classes,              # ['black', 'grizzly', 'teddy'] — order matters
    "arch": "resnet18",
    "img_size": 224,
    "norm_mean": [0.485, 0.456, 0.406],
    "norm_std": [0.229, 0.224, 0.225],
}, "animals_resnet18.pt")